# 07 Modeling Baseline

목적: 05번 최종 모델링 테이블과 06/06b 후보 변수 감사 결과를 사용해, 08번 SHAP/XAI로 넘길 baseline 모델 후보를 선별한다.

이 노트북은 최종 모델 튜닝 노트북이 아니다. 역할은 다음과 같다.

1. feature set을 명시적으로 고정한다.
2. 전체 고객과 프로모션 고객 내부 모델을 분리해 비교한다.
3. 여러 모델 계열을 동일한 split과 동일한 metric으로 비교한다.
4. ROC-AUC뿐 아니라 PR-AUC, churn recall, risk decile lift를 함께 본다.
5. 08번 SHAP에 넘길 best model 후보를 기록한다.

주의:

- 원본 `modeling_feature_table_with_content.csv`에서 컬럼을 물리적으로 삭제하지 않는다.
- 07번 안에서 feature set만 분리한다.
- `is_promotion`, `is_churn_prevented` 변수명은 팀 공유 편의를 위해 유지한다.
- `is_churn_prevented`는 “이번 달 사후 개입 결과”가 아니라 “과거 해지방어 혜택 수혜 이력”으로 해석한다.

In [16]:
from __future__ import annotations

import json
import math
import sys
import warnings
import importlib
from pathlib import Path
from typing import Iterable, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

## 1. 실행 옵션

기본값은 baseline model zoo 실행이다. Optuna는 의도적으로 기본 OFF다. 먼저 baseline을 안정화한 뒤 상위 모델만 튜닝한다.

In [17]:
NOTEBOOK_ID = "07_modeling_baseline"
ID_COL = "membership_row_id"
TARGET_COL = "is_repurchase"
RANDOM_STATE = 42
TEST_SIZE = 0.25
N_DECILES = 10
FIG_DPI = 180

# 기본 실행 옵션
RUN_MODEL_ZOO = True
RUN_OPTUNA = True
RUN_FIGURES = True
SHOW_TABLES = True
SAVE_PREDICTIONS = True
SAVE_FEATURE_IMPORTANCE = True

# 로컬 디버깅용. True로 바꾸면 표본을 줄여 빠르게 smoke test만 한다.
FAST_SMOKE_TEST = False
SMOKE_SAMPLE_N = 2500

# None이면 사용 가능한 전체 model zoo를 실행한다. 예: ["dummy", "logistic", "lightgbm"]
MODEL_NAMES_TO_RUN = None

# 100원딜/프로모션 내부 모델도 실행한다.
RUN_OVERALL_DATASET = True
RUN_PROMOTION_DATASET = True

# 기본 feature set 후보
FEATURE_SET_NAMES_TO_RUN = None  # None이면 정의된 전체 feature set 실행

# Optuna는 07번 후반에서 선택적으로만 사용한다. 기본 OFF.
OPTUNA_N_TRIALS = 40
OPTUNA_TIMEOUT_SEC = 900

## 2. 경로 설정

01~06과 같은 원칙을 따른다. 원천 데이터는 repo root `_data`, 개인 분석 산출물은 `park.ingyeom/reports` 아래에 둔다.

In [18]:
def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]

    for candidate in candidates:
        if (candidate / ".git").exists() and (candidate / "_data").exists():
            return candidate

    for candidate in candidates:
        if candidate.name == "park.ingyeom" and (candidate.parent / "_data").exists():
            return candidate.parent

    raise FileNotFoundError("저장소 루트를 찾지 못했습니다. C:\Code\ott-churn-prediction 또는 그 하위에서 실행하세요.")

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "_data"
WORK_ROOT = PROJECT_ROOT / "park.ingyeom"
REPORTS_DIR = WORK_ROOT / "reports"

INPUT_05_DATA_DIR = REPORTS_DIR / "data" / "05_content_feature_engineering"
INPUT_06_DATA_DIR = REPORTS_DIR / "data" / "06_significance_tests_and_eda"
INPUT_06_REDUNDANCY_DATA_DIR = REPORTS_DIR / "data" / "06_feature_redundancy_audit"
INPUT_06_REDUNDANCY_TABLE_DIR = REPORTS_DIR / "tables" / "06_feature_redundancy_audit"

OUTPUT_DATA_DIR = REPORTS_DIR / "data" / NOTEBOOK_ID
OUTPUT_TABLE_DIR = REPORTS_DIR / "tables" / NOTEBOOK_ID
OUTPUT_FIGURE_DIR = REPORTS_DIR / "figures" / NOTEBOOK_ID
OUTPUT_MODEL_DIR = REPORTS_DIR / "models" / NOTEBOOK_ID

for d in [OUTPUT_DATA_DIR, OUTPUT_TABLE_DIR, OUTPUT_FIGURE_DIR, OUTPUT_MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_05_DATA_DIR:", INPUT_05_DATA_DIR)
print("INPUT_06_DATA_DIR:", INPUT_06_DATA_DIR)
print("INPUT_06_REDUNDANCY_DATA_DIR:", INPUT_06_REDUNDANCY_DATA_DIR)
print("OUTPUT_DATA_DIR:", OUTPUT_DATA_DIR)
print("OUTPUT_TABLE_DIR:", OUTPUT_TABLE_DIR)
print("OUTPUT_FIGURE_DIR:", OUTPUT_FIGURE_DIR)
print("OUTPUT_MODEL_DIR:", OUTPUT_MODEL_DIR)

PROJECT_ROOT: c:\Code\ott-churn-prediction
INPUT_05_DATA_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\05_content_feature_engineering
INPUT_06_DATA_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\06_significance_tests_and_eda
INPUT_06_REDUNDANCY_DATA_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\06_feature_redundancy_audit
OUTPUT_DATA_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\07_modeling_baseline
OUTPUT_TABLE_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\tables\07_modeling_baseline
OUTPUT_FIGURE_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\figures\07_modeling_baseline
OUTPUT_MODEL_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\models\07_modeling_baseline


## 3. 한글 폰트와 환경 요약

시각화 한글 깨짐을 줄이기 위해 가능한 한글 폰트를 탐색한다. 패키지 버전은 재현성 추적용으로 저장한다.

In [19]:
def configure_korean_font() -> dict[str, Any]:
    preferred_fonts = [
        "Malgun Gothic", "AppleGothic", "NanumGothic", "Nanum Gothic",
        "Noto Sans CJK KR", "Noto Sans KR", "Noto Sans CJK", "Arial Unicode MS", "DejaVu Sans",
    ]
    available_names = {f.name for f in font_manager.fontManager.ttflist}
    selected = None
    for name in preferred_fonts:
        if name in available_names:
            selected = name
            break
    if selected is None:
        selected = "DejaVu Sans"
    plt.rcParams["font.family"] = selected
    plt.rcParams["axes.unicode_minus"] = False
    return {"selected_font": selected, "available_font_count": len(available_names)}

font_config = configure_korean_font()

packages = ["pandas", "numpy", "sklearn", "matplotlib", "joblib", "scipy", "lightgbm", "catboost", "xgboost", "shap", "optuna"]
env_rows = [{"package": "python", "status": "OK", "version": sys.version}]
for pkg in packages:
    try:
        module = importlib.import_module(pkg)
        env_rows.append({"package": pkg, "status": "OK", "version": getattr(module, "__version__", "VERSION_ATTR_NOT_FOUND")})
    except Exception as e:
        env_rows.append({"package": pkg, "status": "MISSING_OR_ERROR", "version": f"{type(e).__name__}: {e}"})

env_summary = pd.DataFrame(env_rows)
env_summary.to_csv(OUTPUT_TABLE_DIR / "07_environment_summary.csv", index=False, encoding="utf-8-sig")
font_summary = pd.DataFrame([
    {"metric": "selected_font", "value": font_config["selected_font"]},
    {"metric": "available_font_count", "value": font_config["available_font_count"]},
])
font_summary.to_csv(OUTPUT_TABLE_DIR / "07_font_config.csv", index=False, encoding="utf-8-sig")

display(env_summary)
display(font_summary)

,package,status,version
0,python,OK,"3.10.0 (tags/v3.10.0:b494f59, Oct 4 2021, 19:..."
1,pandas,OK,2.3.3
2,numpy,OK,1.26.4
3,sklearn,OK,1.7.2
4,matplotlib,OK,3.8.4
5,joblib,OK,1.5.3
6,scipy,OK,1.15.3
7,lightgbm,OK,4.6.0
8,catboost,OK,1.2.10
9,xgboost,OK,3.2.0


,metric,value
0,selected_font,Malgun Gothic
1,available_font_count,204


## 4. 입력 파일 로드 및 기본 검산

07번은 raw 데이터를 다시 읽지 않는다. 05번 최종 모델링 테이블과 06/06b 산출물을 입력으로 사용한다.

In [20]:
def first_existing(candidates: Iterable[Path], label: str, required: bool = True) -> Path | None:
    candidates = list(candidates)
    for p in candidates:
        if p.exists():
            return p
    if required:
        raise FileNotFoundError(f"{label} 파일을 찾지 못했습니다. 후보 경로:\n" + "\n".join(str(p) for p in candidates))
    return None

PATH_MODELING_TABLE = first_existing([
    INPUT_05_DATA_DIR / "modeling_feature_table_with_content.csv",
], "05 modeling_feature_table_with_content.csv")

PATH_CANDIDATE_FEATURES = first_existing([
    INPUT_06_DATA_DIR / "06_candidate_features_for_modeling.csv",
], "06_candidate_features_for_modeling.csv")

PATH_ALL_RESULTS = first_existing([
    INPUT_06_DATA_DIR / "06_significance_all_results_with_decision.csv",
], "06_significance_all_results_with_decision.csv", required=False)

PATH_REDUNDANCY_REFINED = first_existing([
    INPUT_06_REDUNDANCY_DATA_DIR / "06_feature_modeling_candidate_refined.csv",
], "06_feature_modeling_candidate_refined.csv")

PATH_REDUNDANCY_REVIEW = first_existing([
    INPUT_06_REDUNDANCY_DATA_DIR / "06_feature_modeling_candidate_redundancy_review.csv",
], "06_feature_modeling_candidate_redundancy_review.csv", required=False)

PATH_MULTICOLLINEARITY_SUMMARY = first_existing([
    INPUT_06_REDUNDANCY_TABLE_DIR / "06_feature_multicollinearity_summary.csv",
], "06_feature_multicollinearity_summary.csv", required=False)

modeling = pd.read_csv(PATH_MODELING_TABLE)
candidate_features = pd.read_csv(PATH_CANDIDATE_FEATURES)
all_results = pd.read_csv(PATH_ALL_RESULTS) if PATH_ALL_RESULTS is not None else pd.DataFrame()
redundancy_refined = pd.read_csv(PATH_REDUNDANCY_REFINED)
redundancy_review = pd.read_csv(PATH_REDUNDANCY_REVIEW) if PATH_REDUNDANCY_REVIEW is not None else pd.DataFrame()
multicollinearity_summary = pd.read_csv(PATH_MULTICOLLINEARITY_SUMMARY) if PATH_MULTICOLLINEARITY_SUMMARY is not None else pd.DataFrame()

if FAST_SMOKE_TEST:
    # stratified smoke sample
    modeling = (
        modeling.groupby(TARGET_COL, group_keys=False)
        .apply(lambda x: x.sample(n=min(len(x), max(1, int(SMOKE_SAMPLE_N * len(x) / len(modeling)))), random_state=RANDOM_STATE))
        .sample(frac=1, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )
    print("FAST_SMOKE_TEST enabled. sampled rows:", len(modeling))

input_file_summary = pd.DataFrame([
    {"name": "modeling_feature_table_with_content", "path": str(PATH_MODELING_TABLE), "rows": len(modeling), "cols": modeling.shape[1], "required": True},
    {"name": "06_candidate_features_for_modeling", "path": str(PATH_CANDIDATE_FEATURES), "rows": len(candidate_features), "cols": candidate_features.shape[1], "required": True},
    {"name": "06_significance_all_results_with_decision", "path": str(PATH_ALL_RESULTS) if PATH_ALL_RESULTS else None, "rows": len(all_results), "cols": all_results.shape[1] if len(all_results) else 0, "required": False},
    {"name": "06_feature_modeling_candidate_refined", "path": str(PATH_REDUNDANCY_REFINED), "rows": len(redundancy_refined), "cols": redundancy_refined.shape[1], "required": True},
    {"name": "06_feature_modeling_candidate_redundancy_review", "path": str(PATH_REDUNDANCY_REVIEW) if PATH_REDUNDANCY_REVIEW else None, "rows": len(redundancy_review), "cols": redundancy_review.shape[1] if len(redundancy_review) else 0, "required": False},
    {"name": "06_feature_multicollinearity_summary", "path": str(PATH_MULTICOLLINEARITY_SUMMARY) if PATH_MULTICOLLINEARITY_SUMMARY else None, "rows": len(multicollinearity_summary), "cols": multicollinearity_summary.shape[1] if len(multicollinearity_summary) else 0, "required": False},
])
input_file_summary.to_csv(OUTPUT_TABLE_DIR / "07_input_file_summary.csv", index=False, encoding="utf-8-sig")
display(input_file_summary)

required_cols = {ID_COL, TARGET_COL}
missing_required = sorted(required_cols - set(modeling.columns))
if missing_required:
    raise ValueError(f"모델링 테이블 필수 컬럼 누락: {missing_required}")
if not modeling[ID_COL].is_unique:
    raise ValueError(f"{ID_COL}가 unique하지 않습니다.")
if sorted(modeling[TARGET_COL].dropna().unique().tolist()) != [0, 1]:
    raise ValueError(f"{TARGET_COL}는 0/1 binary여야 합니다. 현재 값: {sorted(modeling[TARGET_COL].dropna().unique().tolist())}")

basic_summary = pd.DataFrame([
    {"metric": "rows", "value": len(modeling)},
    {"metric": "columns", "value": modeling.shape[1]},
    {"metric": "membership_row_id_unique", "value": bool(modeling[ID_COL].is_unique)},
    {"metric": "repurchase_1_count", "value": int((modeling[TARGET_COL] == 1).sum())},
    {"metric": "repurchase_0_count", "value": int((modeling[TARGET_COL] == 0).sum())},
    {"metric": "repurchase_rate", "value": float(modeling[TARGET_COL].mean())},
])
basic_summary.to_csv(OUTPUT_TABLE_DIR / "07_input_basic_summary.csv", index=False, encoding="utf-8-sig")
display(basic_summary)

,name,path,rows,cols,required
0,modeling_feature_table_with_content,c:\Code\ott-churn-prediction\park.ingyeom\repo...,14922,230,True
1,06_candidate_features_for_modeling,c:\Code\ott-churn-prediction\park.ingyeom\repo...,39,33,True
2,06_significance_all_results_with_decision,c:\Code\ott-churn-prediction\park.ingyeom\repo...,1656,32,False
3,06_feature_modeling_candidate_refined,c:\Code\ott-churn-prediction\park.ingyeom\repo...,39,12,True
4,06_feature_modeling_candidate_redundancy_review,c:\Code\ott-churn-prediction\park.ingyeom\repo...,54,34,False
5,06_feature_multicollinearity_summary,c:\Code\ott-churn-prediction\park.ingyeom\repo...,12,6,False


,metric,value
0,rows,14922
1,columns,230
2,membership_row_id_unique,True
3,repurchase_1_count,10076
4,repurchase_0_count,4846
5,repurchase_rate,0.675245


## 5. Feature set 정책

06b의 기계적 redundancy 결과를 그대로 쓰지 않는다. 팀 커뮤니케이션과 프로젝트 해석을 고려해 대표 변수를 override한다.

정책:

- `is_promotion` 유지, `is_100won` 제외.
- `is_churn_prevented` 유지, `has_prior_churn_prevention_benefit` 제외.
- 원본 컬럼은 삭제하지 않고, 07번 feature set에서만 제외한다.

In [21]:
ID_LIKE_COLS = {
    ID_COL, "USER_KEY", "product_code", "reg_date", "end_date",
}
DATE_LIKE_KEYWORDS = ["week4", "day21", "day22", "day23", "day24", "day25", "day26", "day27"]

# 팀 공유 변수명 유지 정책
MANUAL_KEEP = {
    "is_promotion",
    "is_churn_prevented",
}
MANUAL_DROP = {
    "is_100won",
    "has_prior_churn_prevention_benefit",
    "no_watch_obs_flag",
    "has_usage_feature",
    "no_usage_feature_flag",
    "has_content_feature",
    "no_content_feature_flag",
}

# 해석용 reduced set에서 특히 조심할 중복 변수
REDUCED_DROP = MANUAL_DROP | {
    "price",
    "screen_1_flag", "screen_2_flag", "screen_4_flag",
    "week1_ratio", "week3_ratio", "daily_watch_slope",
    "metadata_missing_watch_ratio", "usable_metadata_watch_ratio",
    "age_band",
}

# 06 후보에서 시작한다.
if "feature" not in redundancy_refined.columns:
    raise ValueError("06_feature_modeling_candidate_refined.csv에 feature 컬럼이 없습니다.")

candidate_base = [f for f in redundancy_refined["feature"].dropna().astype(str).tolist() if f in modeling.columns]

# 06 후보 파일에 없더라도 07에서 비교하고 싶은 핵심 변수.
POLICY_ADD_FEATURES = [
    "is_promotion", "is_churn_prevented", "max_screen", "age", "gender", "billing_method", "payment_device",
    "is_user_verified", "reg_hour", "subscription_days", "has_watch_obs",
    "total_watch_time", "total_sessions", "unique_contents", "unique_days", "watch_days_ratio",
    "days_since_last_watch_to_obs_end", "front_loaded_ratio", "late_ratio", "w3_minus_w1_watch_time",
    "max_day_share", "stable_2screen_active", "discount_sensitive_risk", "family_2screen_lifestyle", "premium_action_trial_risk",
    "metadata_covered_watch_ratio", "genre_tag_entropy_norm", "country_entropy_norm",
    "rating_ratio_전체", "rating_ratio_12세", "rating_ratio_15세", "rating_ratio_청불",
    "tag_ratio_액션", "tag_ratio_드라마", "tag_ratio_애니메이션_키즈", "tag_ratio_가족",
    "avg_runtime_weighted", "is_long_movie_ratio", "is_recent_content_ratio",
]
policy_add = [f for f in POLICY_ADD_FEATURES if f in modeling.columns]

# content broader set: 05번 콘텐츠 비율/태그 계열 일부를 추가.
content_prefixes = (
    "tag_ratio_", "alloc_ratio_", "rating_ratio_", "country_ratio_",
    "is_kids_animation_ratio", "is_family_content_ratio", "is_adult_content_ratio",
    "is_korean_content_ratio", "is_us_content_ratio", "is_japanese_content_ratio",
    "is_recent_content_ratio", "is_old_content_ratio", "is_long_movie_ratio", "is_short_content_ratio",
)
content_broad = [c for c in modeling.columns if c.startswith(content_prefixes)]
content_broad += [c for c in ["genre_tag_entropy_norm", "country_entropy_norm", "avg_runtime_weighted", "avg_release_year_weighted", "avg_content_age_from_2021_weighted"] if c in modeling.columns]
content_broad = sorted(set(content_broad))

# 전체 후보 pool
base_pool = []
for f in candidate_base + policy_add:
    if f not in base_pool:
        base_pool.append(f)

def clean_features(features: list[str], drop: set[str] | None = None, dataset_name: str | None = None) -> list[str]:
    drop = set() if drop is None else set(drop)
    cleaned = []
    for f in features:
        if f not in modeling.columns:
            continue
        if f in ID_LIKE_COLS or f == TARGET_COL:
            continue
        if f in drop:
            continue
        if any(k.lower() in f.lower() for k in DATE_LIKE_KEYWORDS):
            continue
        if dataset_name == "promotion_only" and f == "is_promotion":
            continue
        if f not in cleaned:
            cleaned.append(f)
    # zero variance 제거는 dataset별 split 전에 수행한다.
    return cleaned

# reduced는 해석성과 중복 회피를 우선한다.
core_reduced_seed = [
    "is_promotion", "max_screen", "is_churn_prevented",
    "age", "gender", "billing_method", "payment_device", "is_user_verified", "reg_hour", "subscription_days",
    "has_watch_obs", "days_since_last_watch_to_obs_end",
    "front_loaded_ratio", "late_ratio", "w3_minus_w1_watch_time",
    "unique_days", "watch_days_ratio", "max_day_share",
    "stable_2screen_active", "discount_sensitive_risk", "family_2screen_lifestyle", "premium_action_trial_risk",
    "metadata_covered_watch_ratio", "genre_tag_entropy_norm", "country_entropy_norm",
]
core_reduced = clean_features(core_reduced_seed, REDUCED_DROP)

# tree candidate는 완전 중복 alias와 ID/date만 제거하고 REVIEW도 허용한다.
tree_candidate = clean_features(base_pool, MANUAL_DROP)

# content_added는 tree_candidate에 콘텐츠 비율 변수를 더 넓게 추가한다.
content_added = clean_features(tree_candidate + content_broad, MANUAL_DROP)

# prior 변수 제외 민감도 모델.
without_churn_prevention = clean_features(tree_candidate, MANUAL_DROP | {"is_churn_prevented"})

# 프로모션/요금제 상호작용 중심 비교 세트.
promotion_plan_focused_seed = [
    "is_promotion", "max_screen", "screen_1_flag", "screen_2_flag", "screen_4_flag",
    "promo_x_1screen", "promo_x_2screen", "promo_x_4screen",
    "is_churn_prevented", "age", "gender", "billing_method", "payment_device", "has_watch_obs",
    "front_loaded_ratio", "late_ratio", "w3_minus_w1_watch_time",
    "stable_2screen_active", "discount_sensitive_risk", "family_2screen_lifestyle", "premium_action_trial_risk",
]
promotion_plan_focused = clean_features(promotion_plan_focused_seed, MANUAL_DROP)

# promotion_only에서 해석 가능한 축소 feature set.
# 목적:
# - promotion_only에서 tree_candidate/logistic이 잘 나왔지만, tree_candidate는 해석용으로 다소 복잡함
# - is_promotion, is_churn_prevented 변수명은 팀 커뮤니케이션을 위해 유지
# - price, screen dummy, promo_x_*screen, week ratio 중복 변수는 제외
# - promotion_only dataset에서는 is_promotion이 상수이므로 adapt_features_to_dataset()에서 자동 제외됨

promotion_reduced_interpretable_seed = [
    "is_promotion",
    "max_screen",
    "is_churn_prevented",

    "age",
    "gender",
    "billing_method",
    "payment_device",
    "is_user_verified",
    "subscription_days",

    "has_watch_obs",
    "days_since_last_watch_to_obs_end",
    "front_loaded_ratio",
    "late_ratio",
    "w3_minus_w1_watch_time",
    "unique_days",
    "watch_days_ratio",
    "max_day_share",

    "stable_2screen_active",
    "discount_sensitive_risk",
    "family_2screen_lifestyle",
    "premium_action_trial_risk",

    "metadata_covered_watch_ratio",
    "genre_tag_entropy_norm",
    "country_entropy_norm",
    "rating_ratio_전체",
    "rating_ratio_12세",
    "rating_ratio_15세",
    "rating_ratio_청불",
    "tag_ratio_액션",
    "tag_ratio_드라마",
    "tag_ratio_가족",
    "avg_runtime_weighted",
    "is_long_movie_ratio",
    "is_recent_content_ratio",
]

promotion_reduced_interpretable = clean_features(
    promotion_reduced_interpretable_seed,
    REDUCED_DROP,
)

FEATURE_SETS = {
    "core_reduced": core_reduced,
    "tree_candidate": tree_candidate,
    "content_added": content_added,
    "without_churn_prevention": without_churn_prevention,
    "promotion_plan_focused": promotion_plan_focused,
    "promotion_reduced_interpretable": promotion_reduced_interpretable,
}

if FEATURE_SET_NAMES_TO_RUN is not None:
    FEATURE_SETS = {k: v for k, v in FEATURE_SETS.items() if k in FEATURE_SET_NAMES_TO_RUN}

# dataset별 zero variance 제거용 함수.
def adapt_features_to_dataset(df: pd.DataFrame, features: list[str], dataset_name: str) -> list[str]:
    adapted = clean_features(features, dataset_name=dataset_name)
    final = []
    for f in adapted:
        if f not in df.columns:
            continue
        if df[f].nunique(dropna=True) <= 1:
            continue
        final.append(f)
    return final

feature_set_rows = []
for fs_name, feats in FEATURE_SETS.items():
    for f in feats:
        r = {"feature_set": fs_name, "feature": f}
        if f in set(MANUAL_KEEP):
            r["manual_policy"] = "KEEP"
        elif f in set(MANUAL_DROP):
            r["manual_policy"] = "DROP"
        else:
            r["manual_policy"] = "AUTO"
        feature_set_rows.append(r)
feature_sets_long = pd.DataFrame(feature_set_rows)
feature_set_summary = feature_sets_long.groupby("feature_set", as_index=False).agg(n_features=("feature", "nunique"))

(OUTPUT_DATA_DIR / "07_feature_sets.json").write_text(json.dumps(FEATURE_SETS, ensure_ascii=False, indent=2), encoding="utf-8")
feature_sets_long.to_csv(OUTPUT_TABLE_DIR / "07_feature_sets_long.csv", index=False, encoding="utf-8-sig")
feature_set_summary.to_csv(OUTPUT_TABLE_DIR / "07_feature_set_summary.csv", index=False, encoding="utf-8-sig")

display(feature_set_summary)

,feature_set,n_features
0,content_added,97
1,core_reduced,25
2,promotion_plan_focused,21
3,promotion_reduced_interpretable,34
4,tree_candidate,56
5,without_churn_prevention,55


## 6. 데이터셋 정의와 train/test split

전체 고객 모델과 프로모션 고객 내부 모델을 분리한다. 프로모션 내부 모델에서는 `is_promotion`처럼 상수화된 변수는 자동 제거된다.

In [22]:
DATASETS = {}
if RUN_OVERALL_DATASET:
    DATASETS["overall"] = modeling.copy()
if RUN_PROMOTION_DATASET:
    if "is_promotion" in modeling.columns:
        DATASETS["promotion_only"] = modeling.loc[modeling["is_promotion"] == 1].copy()
    else:
        print("is_promotion 컬럼이 없어 promotion_only dataset을 만들지 않습니다.")

# 너무 작은 dataset 방어
DATASETS = {k: v for k, v in DATASETS.items() if len(v) >= 500 and v[TARGET_COL].nunique() == 2}

split_rows = []
SPLITS = {}
for dataset_name, df in DATASETS.items():
    train_df, test_df = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=df[TARGET_COL],
    )
    SPLITS[dataset_name] = (train_df.reset_index(drop=True), test_df.reset_index(drop=True))
    split_rows.append({
        "dataset": dataset_name,
        "n_total": len(df),
        "n_train": len(train_df),
        "n_test": len(test_df),
        "target_rate_total": float(df[TARGET_COL].mean()),
        "target_rate_train": float(train_df[TARGET_COL].mean()),
        "target_rate_test": float(test_df[TARGET_COL].mean()),
        "train_test_id_overlap": len(set(train_df[ID_COL]) & set(test_df[ID_COL])),
    })

dataset_split_summary = pd.DataFrame(split_rows)
dataset_split_summary.to_csv(OUTPUT_TABLE_DIR / "07_dataset_split_summary.csv", index=False, encoding="utf-8-sig")
display(dataset_split_summary)

,dataset,n_total,n_train,n_test,target_rate_total,target_rate_train,target_rate_test,train_test_id_overlap
0,overall,14922,11191,3731,0.675245,0.675275,0.675154,0
1,promotion_only,8983,6737,2246,0.631749,0.631735,0.631790,0


## 7. 전처리 파이프라인과 모델 zoo 정의

범주형 변수는 OneHotEncoder, 수치형 변수는 median imputation을 사용한다. Logistic Regression만 StandardScaler를 추가한다.

In [23]:
def get_feature_types(df: pd.DataFrame, features: list[str]) -> tuple[list[str], list[str]]:
    numeric_features = []
    categorical_features = []
    for f in features:
        if f not in df.columns:
            continue
        if pd.api.types.is_numeric_dtype(df[f]):
            numeric_features.append(f)
        else:
            categorical_features.append(f)
    return numeric_features, categorical_features

def make_preprocessor(df: pd.DataFrame, features: list[str], scale_numeric: bool = False) -> ColumnTransformer:
    numeric_features, categorical_features = get_feature_types(df, features)
    if scale_numeric:
        numeric_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])
    else:
        numeric_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    transformers = []
    if numeric_features:
        transformers.append(("num", numeric_pipeline, numeric_features))
    if categorical_features:
        transformers.append(("cat", categorical_pipeline, categorical_features))
    return ColumnTransformer(transformers=transformers, remainder="drop", verbose_feature_names_out=False)

def make_pipeline(model_name: str, model: Any, train_df: pd.DataFrame, features: list[str]) -> Pipeline:
    scale_numeric = model_name == "logistic"
    return Pipeline([
        ("preprocess", make_preprocessor(train_df, features, scale_numeric=scale_numeric)),
        ("model", model),
    ])

def import_optional_model(package: str, attr: str):
    try:
        module = importlib.import_module(package)
        return getattr(module, attr), None
    except Exception as e:
        return None, f"{type(e).__name__}: {e}"

LGBMClassifier, lgbm_error = import_optional_model("lightgbm", "LGBMClassifier")
XGBClassifier, xgb_error = import_optional_model("xgboost", "XGBClassifier")
CatBoostClassifier, cat_error = import_optional_model("catboost", "CatBoostClassifier")

MODEL_REGISTRY = {
    "dummy": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
    "logistic": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=None),
    "random_forest": RandomForestClassifier(n_estimators=350, max_depth=None, min_samples_leaf=20, class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1),
    "extra_trees": ExtraTreesClassifier(n_estimators=350, max_depth=None, min_samples_leaf=20, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    "hist_gradient_boosting": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, l2_regularization=0.05, random_state=RANDOM_STATE),
    "gradient_boosting": GradientBoostingClassifier(n_estimators=250, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE),
}
if LGBMClassifier is not None:
    MODEL_REGISTRY["lightgbm"] = LGBMClassifier(
        n_estimators=350, learning_rate=0.04, num_leaves=31, min_child_samples=30,
        subsample=0.85, colsample_bytree=0.85, objective="binary", random_state=RANDOM_STATE,
        n_jobs=-1, verbose=-1,
    )
if XGBClassifier is not None:
    MODEL_REGISTRY["xgboost"] = XGBClassifier(
        n_estimators=350, learning_rate=0.04, max_depth=3, min_child_weight=5,
        subsample=0.85, colsample_bytree=0.85, objective="binary:logistic", eval_metric="logloss",
        tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1,
    )
if CatBoostClassifier is not None:
    MODEL_REGISTRY["catboost"] = CatBoostClassifier(
        iterations=350, learning_rate=0.04, depth=5, loss_function="Logloss", eval_metric="AUC",
        random_seed=RANDOM_STATE, verbose=False, allow_writing_files=False, thread_count=-1,
    )

if MODEL_NAMES_TO_RUN is not None:
    MODEL_REGISTRY = {k: v for k, v in MODEL_REGISTRY.items() if k in MODEL_NAMES_TO_RUN}

model_availability = pd.DataFrame([
    {"model_name": name, "available": True, "note": "registered"} for name in MODEL_REGISTRY
] + [
    {"model_name": "lightgbm", "available": LGBMClassifier is not None, "note": lgbm_error},
    {"model_name": "xgboost", "available": XGBClassifier is not None, "note": xgb_error},
    {"model_name": "catboost", "available": CatBoostClassifier is not None, "note": cat_error},
]).drop_duplicates(subset=["model_name"], keep="first")
model_availability.to_csv(OUTPUT_TABLE_DIR / "07_model_availability.csv", index=False, encoding="utf-8-sig")
display(model_availability)

,model_name,available,note
0,dummy,True,registered
1,logistic,True,registered
2,random_forest,True,registered
3,extra_trees,True,registered
4,hist_gradient_boosting,True,registered
5,gradient_boosting,True,registered
6,lightgbm,True,registered
7,xgboost,True,registered
8,catboost,True,registered


## 8. 평가 함수

is_repurchase=1의 확률을 모델이 출력한다. 운영상 위험도는 `risk_score = 1 - p_repurchase`로 정의한다.

In [24]:
def safe_predict_proba(model: Pipeline, X: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        if proba.ndim == 2 and proba.shape[1] >= 2:
            return proba[:, 1]
    if hasattr(model, "decision_function"):
        score = model.decision_function(X)
        return 1 / (1 + np.exp(-score))
    pred = model.predict(X)
    return pred.astype(float)

def compute_metrics(y_true: np.ndarray, proba_repurchase: np.ndarray, threshold_repurchase: float = 0.5) -> dict[str, Any]:
    y_pred = (proba_repurchase >= threshold_repurchase).astype(int)
    risk_score = 1 - proba_repurchase
    y_churn_true = 1 - y_true
    y_churn_pred = (risk_score >= 0.5).astype(int)

    metrics = {
        "roc_auc": roc_auc_score(y_true, proba_repurchase) if len(np.unique(y_true)) == 2 else np.nan,
        "average_precision": average_precision_score(y_true, proba_repurchase) if len(np.unique(y_true)) == 2 else np.nan,
        "brier_score": brier_score_loss(y_true, proba_repurchase),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_repurchase": precision_score(y_true, y_pred, zero_division=0),
        "recall_repurchase": recall_score(y_true, y_pred, zero_division=0),
        "f1_repurchase": f1_score(y_true, y_pred, zero_division=0),
        "precision_churn": precision_score(y_churn_true, y_churn_pred, zero_division=0),
        "recall_churn": recall_score(y_churn_true, y_churn_pred, zero_division=0),
        "f1_churn": f1_score(y_churn_true, y_churn_pred, zero_division=0),
    }
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    metrics.update({"cm_tn_churn_correct": tn, "cm_fp_churn_as_repurchase": fp, "cm_fn_repurchase_as_churn": fn, "cm_tp_repurchase_correct": tp})
    return metrics

def make_decile_table(df_test: pd.DataFrame, proba_repurchase: np.ndarray, dataset_name: str, feature_set: str, model_name: str) -> pd.DataFrame:
    tmp = df_test[[ID_COL, TARGET_COL]].copy()
    tmp["proba_repurchase"] = proba_repurchase
    tmp["risk_score"] = 1 - tmp["proba_repurchase"]
    tmp["is_churn"] = 1 - tmp[TARGET_COL]
    tmp = tmp.sort_values("risk_score", ascending=False).reset_index(drop=True)
    tmp["risk_rank"] = np.arange(1, len(tmp) + 1)
    tmp["risk_decile"] = pd.qcut(tmp["risk_rank"], q=N_DECILES, labels=False, duplicates="drop") + 1
    dec = tmp.groupby("risk_decile", as_index=False).agg(
        n=(ID_COL, "count"),
        avg_risk_score=("risk_score", "mean"),
        avg_proba_repurchase=("proba_repurchase", "mean"),
        actual_repurchase_rate=(TARGET_COL, "mean"),
        actual_churn_rate=("is_churn", "mean"),
        churn_count=("is_churn", "sum"),
    )
    total_churn = tmp["is_churn"].sum()
    dec["churn_capture_rate"] = dec["churn_count"] / total_churn if total_churn else np.nan
    dec["dataset"] = dataset_name
    dec["feature_set"] = feature_set
    dec["model_name"] = model_name
    return dec[["dataset", "feature_set", "model_name", "risk_decile", "n", "avg_risk_score", "avg_proba_repurchase", "actual_repurchase_rate", "actual_churn_rate", "churn_count", "churn_capture_rate"]]

def get_transformed_feature_names(pipe: Pipeline) -> list[str]:
    try:
        return pipe.named_steps["preprocess"].get_feature_names_out().tolist()
    except Exception:
        return []

def extract_feature_importance(pipe: Pipeline, dataset_name: str, feature_set: str, model_name: str) -> pd.DataFrame:
    model = pipe.named_steps["model"]
    names = get_transformed_feature_names(pipe)
    rows = []
    if hasattr(model, "feature_importances_"):
        vals = np.asarray(model.feature_importances_, dtype=float)
        if len(names) == len(vals):
            rows = [{"transformed_feature": n, "importance": float(v), "importance_type": "feature_importances_"} for n, v in zip(names, vals)]
    elif hasattr(model, "coef_"):
        vals = np.asarray(model.coef_).ravel()
        if len(names) == len(vals):
            rows = [{"transformed_feature": n, "importance": float(v), "importance_abs": float(abs(v)), "importance_type": "coef"} for n, v in zip(names, vals)]
    if not rows:
        return pd.DataFrame(columns=["dataset", "feature_set", "model_name", "transformed_feature", "importance", "importance_abs", "importance_type"])
    out = pd.DataFrame(rows)
    if "importance_abs" not in out.columns:
        out["importance_abs"] = out["importance"].abs()
    out["dataset"] = dataset_name
    out["feature_set"] = feature_set
    out["model_name"] = model_name
    return out[["dataset", "feature_set", "model_name", "transformed_feature", "importance", "importance_abs", "importance_type"]].sort_values("importance_abs", ascending=False)

## 9. Baseline model zoo 실행

동일 split에서 dataset × feature set × model을 비교한다. 실패한 모델은 전체 노트북을 죽이지 않고 실패 로그로 남긴다.

In [25]:
metrics_rows = []
prediction_frames = []
decile_frames = []
importance_frames = []
failed_rows = []

for dataset_name, (train_df, test_df) in SPLITS.items():
    y_train = train_df[TARGET_COL].astype(int).to_numpy()
    y_test = test_df[TARGET_COL].astype(int).to_numpy()

    for feature_set_name, raw_features in FEATURE_SETS.items():
        features = adapt_features_to_dataset(train_df, raw_features, dataset_name=dataset_name)
        if len(features) == 0:
            failed_rows.append({"dataset": dataset_name, "feature_set": feature_set_name, "model_name": "ALL", "error": "empty feature set after adaptation"})
            continue

        X_train = train_df[features].copy()
        X_test = test_df[features].copy()

        for model_name, model_obj in MODEL_REGISTRY.items():
            try:
                pipe = make_pipeline(model_name, clone(model_obj), train_df, features)
                pipe.fit(X_train, y_train)
                proba = safe_predict_proba(pipe, X_test)
                metric = compute_metrics(y_test, proba)
                metric.update({
                    "dataset": dataset_name,
                    "feature_set": feature_set_name,
                    "model_name": model_name,
                    "n_train": len(train_df),
                    "n_test": len(test_df),
                    "n_features_original": len(features),
                    "target_rate_train": float(np.mean(y_train)),
                    "target_rate_test": float(np.mean(y_test)),
                })
                metrics_rows.append(metric)

                pred = test_df[[ID_COL, TARGET_COL]].copy()
                pred["dataset"] = dataset_name
                pred["feature_set"] = feature_set_name
                pred["model_name"] = model_name
                pred["proba_repurchase"] = proba
                pred["risk_score"] = 1 - proba
                prediction_frames.append(pred)

                decile_frames.append(make_decile_table(test_df, proba, dataset_name, feature_set_name, model_name))

                if SAVE_FEATURE_IMPORTANCE:
                    imp = extract_feature_importance(pipe, dataset_name, feature_set_name, model_name)
                    if len(imp):
                        importance_frames.append(imp)

            except Exception as e:
                failed_rows.append({
                    "dataset": dataset_name,
                    "feature_set": feature_set_name,
                    "model_name": model_name,
                    "error": f"{type(e).__name__}: {e}",
                })
                print("MODEL FAILED:", dataset_name, feature_set_name, model_name, type(e).__name__, e)

model_metrics = pd.DataFrame(metrics_rows)
prediction_scores = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else pd.DataFrame()
risk_decile_table = pd.concat(decile_frames, ignore_index=True) if decile_frames else pd.DataFrame()
feature_importance = pd.concat(importance_frames, ignore_index=True) if importance_frames else pd.DataFrame()
failed_models = pd.DataFrame(failed_rows)

model_metrics = model_metrics.sort_values(["dataset", "roc_auc", "average_precision"], ascending=[True, False, False]) if len(model_metrics) else model_metrics

model_metrics.to_csv(OUTPUT_DATA_DIR / "07_model_metrics.csv", index=False, encoding="utf-8-sig")
risk_decile_table.to_csv(OUTPUT_DATA_DIR / "07_risk_decile_table.csv", index=False, encoding="utf-8-sig")
if SAVE_PREDICTIONS:
    prediction_scores.to_csv(OUTPUT_DATA_DIR / "07_prediction_scores.csv", index=False, encoding="utf-8-sig")
if SAVE_FEATURE_IMPORTANCE:
    feature_importance.to_csv(OUTPUT_DATA_DIR / "07_feature_importance.csv", index=False, encoding="utf-8-sig")
failed_models.to_csv(OUTPUT_TABLE_DIR / "07_failed_models.csv", index=False, encoding="utf-8-sig")

metrics_summary = model_metrics.copy()
metrics_summary.to_csv(OUTPUT_TABLE_DIR / "07_model_metrics_summary.csv", index=False, encoding="utf-8-sig")

print("models evaluated:", len(model_metrics))
print("failed models:", len(failed_models))
if SHOW_TABLES:
    display(model_metrics.head(30))
    display(failed_models)
else:
    display(model_metrics.head(15))

models evaluated: 108
failed models: 0


,roc_auc,average_precision,brier_score,accuracy,balanced_accuracy,precision_repurchase,recall_repurchase,f1_repurchase,precision_churn,recall_churn,f1_churn,cm_tn_churn_correct,cm_fp_churn_as_repurchase,cm_fn_repurchase_as_churn,cm_tp_repurchase_correct,dataset,feature_set,model_name,n_train,n_test,n_features_original,target_rate_train,target_rate_test
49,0.649103,0.776494,0.205972,0.686947,0.548975,0.698735,0.942834,0.802636,0.566265,0.155116,0.243523,188,1024,144,2375,overall,promotion_reduced_interpretable,hist_gradient_boosting,11191,3731,34,0.675275,0.675154
41,0.647878,0.773478,0.206094,0.685607,0.557829,0.703816,0.922588,0.798488,0.545455,0.193069,0.285192,234,978,195,2324,overall,promotion_plan_focused,gradient_boosting,11191,3731,21,0.675275,0.675154
44,0.647812,0.773982,0.206347,0.682927,0.553061,0.701326,0.923779,0.797327,0.535109,0.182343,0.272000,221,991,192,2327,overall,promotion_plan_focused,catboost,11191,3731,21,0.675275,0.675154
43,0.647182,0.772508,0.206300,0.683195,0.555400,0.702637,0.920206,0.796837,0.534722,0.190594,0.281022,231,981,201,2318,overall,promotion_plan_focused,xgboost,11191,3731,21,0.675275,0.675154
8,0.646754,0.773876,0.206112,0.682123,0.547757,0.698422,0.931322,0.798231,0.534946,0.164191,0.251263,199,1013,173,2346,overall,core_reduced,catboost,11191,3731,25,0.675275,0.675154
7,0.646562,0.772662,0.206021,0.683731,0.554085,0.701839,0.924176,0.797807,0.538647,0.183993,0.274293,223,989,191,2328,overall,core_reduced,xgboost,11191,3731,25,0.675275,0.675154
17,0.646439,0.774312,0.206083,0.685071,0.551224,0.700119,0.933307,0.800068,0.549598,0.169142,0.258675,205,1007,168,2351,overall,tree_candidate,catboost,11191,3731,56,0.675275,0.675154
25,0.646290,0.775407,0.206304,0.684535,0.550399,0.699702,0.933307,0.799796,0.547170,0.167492,0.256475,203,1009,168,2351,overall,content_added,xgboost,11191,3731,97,0.675275,0.675154
22,0.646215,0.775072,0.205912,0.690432,0.548987,0.698487,0.952759,0.806045,0.596610,0.145215,0.233577,176,1036,119,2400,overall,content_added,hist_gradient_boosting,11191,3731,97,0.675275,0.675154
14,0.645832,0.776804,0.206253,0.684535,0.552540,0.700898,0.929337,0.799112,0.544757,0.175743,0.265752,213,999,178,2341,overall,tree_candidate,gradient_boosting,11191,3731,56,0.675275,0.675154


""


## 10. 모델 비교 및 best model 후보 선정

AUC만 보지 않고, PR-AUC, churn recall, decile 상위 위험군의 실제 이탈률도 함께 본다.

In [26]:
if len(model_metrics) == 0:
    raise RuntimeError("성공한 모델이 없습니다. failed_models를 확인하세요.")

# decile 1은 가장 위험하다고 예측된 집단.
top_decile = risk_decile_table.loc[risk_decile_table["risk_decile"] == 1].copy()
top2_decile = risk_decile_table.loc[risk_decile_table["risk_decile"].isin([1, 2])].groupby(["dataset", "feature_set", "model_name"], as_index=False).agg(
    top20_n=("n", "sum"),
    top20_churn_count=("churn_count", "sum"),
    top20_avg_risk_score=("avg_risk_score", "mean"),
)
top2_decile["top20_actual_churn_rate"] = top2_decile["top20_churn_count"] / top2_decile["top20_n"]

comparison = model_metrics.merge(
    top_decile[["dataset", "feature_set", "model_name", "actual_churn_rate", "churn_capture_rate"]].rename(columns={"actual_churn_rate": "top10_actual_churn_rate", "churn_capture_rate": "top10_churn_capture_rate"}),
    on=["dataset", "feature_set", "model_name"],
    how="left",
).merge(
    top2_decile[["dataset", "feature_set", "model_name", "top20_actual_churn_rate"]],
    on=["dataset", "feature_set", "model_name"],
    how="left",
)

# 모델 선택 점수. 절대적 진리가 아니라 baseline 후보 선정을 위한 정렬 기준이다.
comparison["selection_score"] = (
    comparison["roc_auc"].fillna(0) * 0.45
    + comparison["average_precision"].fillna(0) * 0.25
    + comparison["recall_churn"].fillna(0) * 0.15
    + comparison["top10_actual_churn_rate"].fillna(0) * 0.15
)
comparison = comparison.sort_values(["dataset", "selection_score", "roc_auc", "top10_actual_churn_rate"], ascending=[True, False, False, False])
comparison.to_csv(OUTPUT_TABLE_DIR / "07_model_comparison_for_report.csv", index=False, encoding="utf-8-sig")

best_by_dataset = comparison.groupby("dataset", as_index=False).head(1).copy()
best_by_dataset.to_csv(OUTPUT_TABLE_DIR / "07_best_model_by_dataset.csv", index=False, encoding="utf-8-sig")

best_overall = best_by_dataset.iloc[0].to_dict()
if "overall" in set(best_by_dataset["dataset"]):
    best_overall = best_by_dataset.loc[best_by_dataset["dataset"] == "overall"].iloc[0].to_dict()

best_config = {
    "selection_rule": "max selection_score = 0.45*roc_auc + 0.25*average_precision + 0.15*recall_churn + 0.15*top10_actual_churn_rate",
    "best_by_dataset": best_by_dataset.to_dict(orient="records"),
    "recommended_for_08_shap": {
        "dataset": best_overall["dataset"],
        "feature_set": best_overall["feature_set"],
        "model_name": best_overall["model_name"],
    },
}
(OUTPUT_DATA_DIR / "07_best_model_config.json").write_text(json.dumps(best_config, ensure_ascii=False, indent=2), encoding="utf-8")

decile_summary = risk_decile_table.sort_values(["dataset", "feature_set", "model_name", "risk_decile"])
decile_summary.to_csv(OUTPUT_TABLE_DIR / "07_decile_summary.csv", index=False, encoding="utf-8-sig")

display(comparison.head(20))
display(best_by_dataset)

,roc_auc,average_precision,brier_score,accuracy,balanced_accuracy,precision_repurchase,recall_repurchase,f1_repurchase,precision_churn,recall_churn,f1_churn,cm_tn_churn_correct,cm_fp_churn_as_repurchase,cm_fn_repurchase_as_churn,cm_tp_repurchase_correct,dataset,feature_set,model_name,n_train,n_test,n_features_original,target_rate_train,target_rate_test,top10_actual_churn_rate,top10_churn_capture_rate,top20_actual_churn_rate,selection_score
17,0.643979,0.775612,0.231428,0.627714,0.611491,0.758700,0.657801,0.704657,0.442793,0.565182,0.496557,685,527,862,1657,overall,tree_candidate,extra_trees,11191,3731,56,0.675275,0.675154,0.553476,0.170792,0.497992,0.651492
21,0.642803,0.775891,0.233926,0.610828,0.601769,0.754654,0.627630,0.685306,0.426650,0.575908,0.490169,698,514,938,1581,overall,promotion_plan_focused,extra_trees,11191,3731,21,0.675275,0.675154,0.540107,0.166667,0.510040,0.650636
11,0.645690,0.773684,0.229910,0.624229,0.605486,0.753518,0.658992,0.703092,0.437827,0.551980,0.488321,669,543,859,1660,overall,promotion_plan_focused,random_forest,11191,3731,21,0.675275,0.675154,0.553476,0.170792,0.516734,0.649800
14,0.644287,0.775155,0.231254,0.631466,0.611273,0.756963,0.668916,0.710221,0.445847,0.553630,0.493927,671,541,834,1685,overall,content_added,extra_trees,11191,3731,97,0.675275,0.675154,0.553476,0.170792,0.495315,0.649784
19,0.643300,0.774072,0.233012,0.621549,0.607782,0.757083,0.647082,0.697774,0.436629,0.568482,0.493907,689,523,889,1630,overall,core_reduced,extra_trees,11191,3731,25,0.675275,0.675154,0.540107,0.166667,0.495315,0.649291
12,0.645193,0.775611,0.227502,0.636023,0.609939,0.753826,0.684399,0.717437,0.449446,0.535479,0.488705,649,563,795,1724,overall,core_reduced,random_forest,11191,3731,25,0.675275,0.675154,0.553476,0.170792,0.507363,0.647583
33,0.638466,0.769899,0.236062,0.602787,0.598383,0.754042,0.610957,0.675000,0.420118,0.585809,0.489318,710,502,980,1539,overall,tree_candidate,logistic,11191,3731,56,0.675275,0.675154,0.526738,0.162541,0.499331,0.646666
26,0.639987,0.771330,0.232992,0.621549,0.602003,0.750793,0.657801,0.701227,0.434383,0.546205,0.483918,662,550,862,1657,overall,promotion_reduced_interpretable,extra_trees,11191,3731,34,0.675275,0.675154,0.556150,0.171617,0.496653,0.646180
39,0.636072,0.764949,0.236550,0.604664,0.600628,0.755882,0.612148,0.676464,0.422235,0.589109,0.491905,714,498,977,1542,overall,content_added,logistic,11191,3731,97,0.675275,0.675154,0.532086,0.164191,0.491299,0.645649
34,0.638398,0.771360,0.232678,0.619137,0.605353,0.755349,0.644700,0.695652,0.433903,0.566007,0.491228,686,526,895,1624,overall,without_churn_prevention,extra_trees,11191,3731,55,0.675275,0.675154,0.534759,0.165017,0.488621,0.645234


,roc_auc,average_precision,brier_score,accuracy,balanced_accuracy,precision_repurchase,recall_repurchase,f1_repurchase,precision_churn,recall_churn,f1_churn,cm_tn_churn_correct,cm_fp_churn_as_repurchase,cm_fn_repurchase_as_churn,cm_tp_repurchase_correct,dataset,feature_set,model_name,n_train,n_test,n_features_original,target_rate_train,target_rate_test,top10_actual_churn_rate,top10_churn_capture_rate,top20_actual_churn_rate,selection_score
17,0.643979,0.775612,0.231428,0.627714,0.611491,0.758700,0.657801,0.704657,0.442793,0.565182,0.496557,685,527,862,1657,overall,tree_candidate,extra_trees,11191,3731,56,0.675275,0.675154,0.553476,0.170792,0.497992,0.651492
54,0.661251,0.761735,0.229567,0.625111,0.620327,0.733603,0.638478,0.682743,0.492582,0.602177,0.541893,498,329,513,906,promotion_only,tree_candidate,logistic,6737,2246,53,0.631735,0.631790,0.617778,0.168077,0.537778,0.670990


## 11. 기본 시각화

각 dataset별 best model 후보에 대해 ROC, PR, risk decile, feature importance를 저장한다.

In [27]:
def savefig(name: str):
    path = OUTPUT_FIGURE_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close()
    return path

if RUN_FIGURES and len(best_by_dataset):
    for _, row in best_by_dataset.iterrows():
        dataset_name = row["dataset"]
        feature_set_name = row["feature_set"]
        model_name = row["model_name"]
        pred = prediction_scores.loc[
            (prediction_scores["dataset"] == dataset_name) &
            (prediction_scores["feature_set"] == feature_set_name) &
            (prediction_scores["model_name"] == model_name)
        ].copy()
        if len(pred) == 0:
            continue
        y_true = pred[TARGET_COL].astype(int).to_numpy()
        proba = pred["proba_repurchase"].to_numpy()
        risk = pred["risk_score"].to_numpy()

        # ROC
        fpr, tpr, _ = roc_curve(y_true, proba)
        plt.figure(figsize=(7, 6))
        plt.plot(fpr, tpr, label=f"AUC={roc_auc_score(y_true, proba):.3f}")
        plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"ROC Curve | {dataset_name} | {model_name}")
        plt.legend()
        savefig(f"07_roc_curve_{dataset_name}_{feature_set_name}_{model_name}.png")

        # PR
        precision, recall, _ = precision_recall_curve(y_true, proba)
        plt.figure(figsize=(7, 6))
        plt.plot(recall, precision, label=f"AP={average_precision_score(y_true, proba):.3f}")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title(f"Precision-Recall Curve | {dataset_name} | {model_name}")
        plt.legend()
        savefig(f"07_pr_curve_{dataset_name}_{feature_set_name}_{model_name}.png")

        # Risk decile
        dec = risk_decile_table.loc[
            (risk_decile_table["dataset"] == dataset_name) &
            (risk_decile_table["feature_set"] == feature_set_name) &
            (risk_decile_table["model_name"] == model_name)
        ].copy()
        if len(dec):
            plt.figure(figsize=(9, 5))
            plt.bar(dec["risk_decile"].astype(str), dec["actual_churn_rate"])
            plt.xlabel("Risk decile (1 = highest predicted churn risk)")
            plt.ylabel("Actual churn rate")
            plt.title(f"Risk decile actual churn rate | {dataset_name} | {model_name}")
            for x, y, n in zip(dec["risk_decile"].astype(str), dec["actual_churn_rate"], dec["n"]):
                plt.text(x, y, f"{y:.1%}\nn={int(n)}", ha="center", va="bottom", fontsize=8)
            savefig(f"07_risk_decile_{dataset_name}_{feature_set_name}_{model_name}.png")

        # Feature importance
        imp = feature_importance.loc[
            (feature_importance["dataset"] == dataset_name) &
            (feature_importance["feature_set"] == feature_set_name) &
            (feature_importance["model_name"] == model_name)
        ].head(30).copy()
        if len(imp):
            plt.figure(figsize=(11, max(6, 0.32 * len(imp))))
            plt.barh(imp["transformed_feature"][::-1], imp["importance_abs"][::-1])
            plt.xlabel("Absolute importance")
            plt.title(f"Top feature importance | {dataset_name} | {model_name}")
            savefig(f"07_feature_importance_{dataset_name}_{feature_set_name}_{model_name}.png")

figure_files = sorted([p.name for p in OUTPUT_FIGURE_DIR.glob("07_*.png")])
figure_summary = pd.DataFrame({"figure_file": figure_files})
figure_summary.to_csv(OUTPUT_TABLE_DIR / "07_figure_summary.csv", index=False, encoding="utf-8-sig")
print("figures created:", len(figure_files))

figures created: 8


## 12. Optional Optuna 준비 셀

기본값은 OFF다. baseline 결과에서 상위 모델 1~2개를 확인한 뒤, 필요한 경우에만 별도 실행한다.

In [28]:
optuna_status = {"run_optuna": RUN_OPTUNA, "status": "SKIPPED", "note": "RUN_OPTUNA=False"}
if RUN_OPTUNA:
    try:
        import optuna  # type: ignore
        optuna_status = {"run_optuna": RUN_OPTUNA, "status": "READY", "note": "Optuna is installed. Tuning cell is intentionally left as extension point."}
        # 실제 튜닝은 baseline 결과를 본 뒤 상위 1~2개 모델만 대상으로 별도 확장하는 것을 권장한다.
    except Exception as e:
        optuna_status = {"run_optuna": RUN_OPTUNA, "status": "MISSING_OR_ERROR", "note": f"{type(e).__name__}: {e}"}

optuna_summary = pd.DataFrame([optuna_status])
optuna_summary.to_csv(OUTPUT_TABLE_DIR / "07_optuna_status.csv", index=False, encoding="utf-8-sig")
display(optuna_summary)

,run_optuna,status,note
0,True,READY,Optuna is installed. Tuning cell is intentiona...


## 13. 최종 검산

07번은 모델링 결과가 생성되었는지뿐 아니라, 피처셋과 split, 산출물 구조가 안전한지도 확인한다.

In [29]:
expected_data_files = [
    OUTPUT_DATA_DIR / "07_feature_sets.json",
    OUTPUT_DATA_DIR / "07_model_metrics.csv",
    OUTPUT_DATA_DIR / "07_risk_decile_table.csv",
    OUTPUT_DATA_DIR / "07_prediction_scores.csv",
    OUTPUT_DATA_DIR / "07_feature_importance.csv",
    OUTPUT_DATA_DIR / "07_best_model_config.json",
]
expected_table_files = [
    OUTPUT_TABLE_DIR / "07_environment_summary.csv",
    OUTPUT_TABLE_DIR / "07_font_config.csv",
    OUTPUT_TABLE_DIR / "07_input_file_summary.csv",
    OUTPUT_TABLE_DIR / "07_input_basic_summary.csv",
    OUTPUT_TABLE_DIR / "07_feature_sets_long.csv",
    OUTPUT_TABLE_DIR / "07_feature_set_summary.csv",
    OUTPUT_TABLE_DIR / "07_dataset_split_summary.csv",
    OUTPUT_TABLE_DIR / "07_model_availability.csv",
    OUTPUT_TABLE_DIR / "07_model_metrics_summary.csv",
    OUTPUT_TABLE_DIR / "07_model_comparison_for_report.csv",
    OUTPUT_TABLE_DIR / "07_best_model_by_dataset.csv",
    OUTPUT_TABLE_DIR / "07_decile_summary.csv",
    OUTPUT_TABLE_DIR / "07_failed_models.csv",
    OUTPUT_TABLE_DIR / "07_figure_summary.csv",
    OUTPUT_TABLE_DIR / "07_optuna_status.csv",
]

all_feature_values = [f for feats in FEATURE_SETS.values() for f in feats]
duplicate_feature_sets = []
for name, feats in FEATURE_SETS.items():
    if len(feats) != len(set(feats)):
        duplicate_feature_sets.append(name)

fourth_week_like_features = [f for f in all_feature_values if any(k.lower() in f.lower() for k in DATE_LIKE_KEYWORDS)]
manual_drop_used = sorted(set(all_feature_values) & MANUAL_DROP)
# is_promotion/is_churn_prevented 정책 확인
manual_keep_missing = sorted([f for f in MANUAL_KEEP if f in modeling.columns and f not in all_feature_values])

final_checks = pd.DataFrame([
    {"check": "project_root_is_repo_root", "value": str(PROJECT_ROOT), "pass": (PROJECT_ROOT / ".git").exists()},
    {"check": "data_root_is_repo_data", "value": str(DATA_ROOT), "pass": DATA_ROOT.exists()},
    {"check": "reports_dir_is_park_reports", "value": str(REPORTS_DIR), "pass": REPORTS_DIR == WORK_ROOT / "reports"},
    {"check": "input_05_data_dir_is_reports_data_05", "value": str(INPUT_05_DATA_DIR), "pass": INPUT_05_DATA_DIR == REPORTS_DIR / "data" / "05_content_feature_engineering"},
    {"check": "input_rows_is_14922_or_smoke", "value": len(modeling), "pass": (len(modeling) == 14922) or FAST_SMOKE_TEST},
    {"check": "membership_row_id_unique", "value": bool(modeling[ID_COL].is_unique), "pass": bool(modeling[ID_COL].is_unique)},
    {"check": "target_is_binary_0_1", "value": sorted(modeling[TARGET_COL].dropna().unique().tolist()), "pass": sorted(modeling[TARGET_COL].dropna().unique().tolist()) == [0, 1]},
    {"check": "feature_sets_not_empty", "value": feature_set_summary.to_dict(orient="records"), "pass": all(len(v) > 0 for v in FEATURE_SETS.values())},
    {"check": "no_duplicate_features_in_each_set", "value": duplicate_feature_sets, "pass": len(duplicate_feature_sets) == 0},
    {"check": "no_fourth_week_feature_detected", "value": fourth_week_like_features, "pass": len(fourth_week_like_features) == 0},
    {"check": "manual_drop_features_not_used", "value": manual_drop_used, "pass": len(manual_drop_used) == 0},
    {"check": "manual_keep_features_present", "value": manual_keep_missing, "pass": len(manual_keep_missing) == 0},
    {"check": "train_test_no_membership_row_overlap", "value": dataset_split_summary[["dataset", "train_test_id_overlap"]].to_dict(orient="records"), "pass": bool((dataset_split_summary["train_test_id_overlap"] == 0).all())},
    {"check": "models_trained_gt_zero", "value": len(model_metrics), "pass": len(model_metrics) > 0},
    {"check": "metrics_file_created", "value": str(OUTPUT_DATA_DIR / "07_model_metrics.csv"), "pass": (OUTPUT_DATA_DIR / "07_model_metrics.csv").exists()},
    {"check": "prediction_scores_created", "value": str(OUTPUT_DATA_DIR / "07_prediction_scores.csv"), "pass": (OUTPUT_DATA_DIR / "07_prediction_scores.csv").exists()},
    {"check": "decile_table_created", "value": str(OUTPUT_DATA_DIR / "07_risk_decile_table.csv"), "pass": (OUTPUT_DATA_DIR / "07_risk_decile_table.csv").exists()},
    {"check": "feature_importance_created", "value": str(OUTPUT_DATA_DIR / "07_feature_importance.csv"), "pass": (OUTPUT_DATA_DIR / "07_feature_importance.csv").exists()},
    {"check": "best_model_config_created", "value": str(OUTPUT_DATA_DIR / "07_best_model_config.json"), "pass": (OUTPUT_DATA_DIR / "07_best_model_config.json").exists()},
    {"check": "figures_created_if_enabled", "value": len(figure_files), "pass": (not RUN_FIGURES) or len(figure_files) > 0},
    {"check": "data_outputs_exist", "value": [p.name for p in expected_data_files if p.exists()], "pass": all(p.exists() for p in expected_data_files)},
    {"check": "table_outputs_exist", "value": [p.name for p in expected_table_files if p.exists()], "pass": all(p.exists() for p in expected_table_files)},
])

final_checks.to_csv(OUTPUT_TABLE_DIR / "07_final_checks.csv", index=False, encoding="utf-8-sig")
display(final_checks)

if not final_checks["pass"].all():
    failed = final_checks.loc[~final_checks["pass"]]
    raise AssertionError(f"07_modeling_baseline 최종 검산 실패:\n{failed}")

print("07_modeling_baseline passed final checks.")

,check,value,pass
0,project_root_is_repo_root,c:\Code\ott-churn-prediction,True
1,data_root_is_repo_data,c:\Code\ott-churn-prediction\_data,True
2,reports_dir_is_park_reports,c:\Code\ott-churn-prediction\park.ingyeom\reports,True
3,input_05_data_dir_is_reports_data_05,c:\Code\ott-churn-prediction\park.ingyeom\repo...,True
4,input_rows_is_14922_or_smoke,14922,True
5,membership_row_id_unique,True,True
6,target_is_binary_0_1,"[0, 1]",True
7,feature_sets_not_empty,"[{'feature_set': 'content_added', 'n_features'...",True
8,no_duplicate_features_in_each_set,[],True
9,no_fourth_week_feature_detected,[],True


07_modeling_baseline passed final checks.


In [30]:
print("FEATURE_SETS keys:")
for k, v in FEATURE_SETS.items():
    print(k, len(v), v[:5])

assert "promotion_reduced_interpretable" in FEATURE_SETS
assert len(FEATURE_SETS["promotion_reduced_interpretable"]) > 0

print("promotion_reduced_interpretable OK")

FEATURE_SETS keys:
core_reduced 25 ['is_promotion', 'max_screen', 'is_churn_prevented', 'age', 'gender']
tree_candidate 56 ['is_promotion', 'is_churn_prevented', 'daily_watch_slope', 'alloc_dur_코미디', 'max_screen']
content_added 97 ['is_promotion', 'is_churn_prevented', 'daily_watch_slope', 'alloc_dur_코미디', 'max_screen']
without_churn_prevention 55 ['is_promotion', 'daily_watch_slope', 'alloc_dur_코미디', 'max_screen', 'promo_x_4screen']
promotion_plan_focused 21 ['is_promotion', 'max_screen', 'screen_1_flag', 'screen_2_flag', 'screen_4_flag']
promotion_reduced_interpretable 34 ['is_promotion', 'max_screen', 'is_churn_prevented', 'age', 'gender']
promotion_reduced_interpretable OK
